# បណ្តាញបង្កើត

បណ្តាញប្រស่า (Recurrent Neural Networks - RNNs) និងបែបបទកោសិកាដែលមានទ្វារដូចជា ឧក្រិដ្ឋជាតិសម័យបណ្តោះអាសន្នបែបវែង (Long Short Term Memory Cells - LSTMs) និងឧក្រិដ្ឋជាតិសម័យជាតិត្រកូល (Gated Recurrent Units - GRUs) ផ្តល់មកនូវយន្តការសម្រាប់ការម៉ូដែលភាសា ដែលមានន័យថា ពួកគេអាចរៀនលំដាប់ពាក្យ និងផ្តល់ការព្យាករណ៍សម្រាប់ពាក្យបន្ទាប់ក្នុងលំដាប់។ នេះអនុញ្ញាតឲ្យយើងប្រើ RNN សម្រាប់ **ភារកិច្ចបង្កើត** ដូចជា ការបង្កើតអត្ថបទទូទៅ ការបកប្រែម៉ាស៊ីន និងឯងការបណ្តឹងរូបភាព។

នៅក្នុងគ្រឹះសម្រាប់ការរចនាបណ្តាញ RNN ដែលយើងបានពិភាក្សានៅឯកត្តាមុន រាល់អង្គភាព RNN ក៏បានបង្កើតស្ថានភាពលាក់បន្ទាប់ជាលទ្ធផល។ ទោះជាយ៉ាងណា យើងក៏អាចបន្ថែមលទ្ធផលមួយទៀតទៅរាល់អង្គភាពប្រសាទដែលអាចអោយយើងបញ្ចេញជាលំដាប់ (ដែលមានប្រវែងស្មើនឹងលំដាប់ដើម)។ លើសពីនេះ យើងអាចប្រើអង្គភាព RNN ដែលមិនទទួលអ្នកបញ្ចូលនៅរាល់ជំហានទេ ហើយគ្រាន់តែទទួលវ៉ិចទ័រស្ថានភាពដំបូង ហើយបន្ទាប់មកបង្កើតលំដាប់លទ្ធផល។

នៅក្នុងសៀវភៅកំណត់ត្រានេះ យើងនឹងផ្តោតទៅលើម៉ូដែលបង្កើតបណ្តាក់សាមញ្ញ ដែលជួយយើងបង្កើតអត្ថបទ។ ដើម្បីសម្រួល អាចសង់បណ្តាញ **កម្រិតតួអក្សរ** ដែលបង្កើតអត្ថបទតួអក្សរតាមតួអក្សរ។ នៅពេលហ្វឹកហាត់ យើងត្រូវយកអត្ថបទមួយ ហើយបំបែកវាទៅជា លំដាប់តួអក្សរ។


In [1]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

ds_train, ds_test = tfds.load('ag_news_subset').values()

## ការបង្កើតវាក្យសព្ទតួអក្សរ

ដើម្បីបង្កើតបណ្តាញបង្កើតនៅតួអក្សររង្វែង, យើងត្រូវបំបែកអត្ថបទទៅជាតួអក្សរដូចខ្លួនតូចមួយមិនមែនជាពាក្យទេ។ ស្រទាប់ `TextVectorization` ដែលយើងបានប្រើមុនមិនអាចធ្វើបែបនេះបានទេ, ដូចនេះយើងមានជម្រើសពីរម៉េ៖

* ដាក់អត្ថបទដោយដៃហើយធ្វើការកំណត់និយមន័យពាក្យ 'ដោយដៃ', ដូចដែលមាននៅក្នុង [គំរូផ្លូវការនៃ Keras](https://keras.io/examples/generative/lstm_character_level_text_generation/)
* ប្រើថ្នាក់ `Tokenizer` សម្រាប់កំណត់និយមន័យតួអក្សររង្វែង។

យើងនឹងជ្រើសរើសជម្រើសទីពីរ។ អាចប្រើ `Tokenizer` សម្រាប់កំណត់និយមន័យទៅជាពាក្យបានផងដែរ, ដូច្នេះអ្នកអាចប្ដូរពីការកំណត់និយមន័យតួអក្សរទៅកាន់ការកំណត់និយមន័យពាក្យបានយ៉ាងងាយស្រួល។

ដើម្បីធ្វើការកំណត់និយមន័យតួអក្សររង្វែង, យើងត្រូវផ្តល់ប៉ារ៉ាម៉ែត្រ `char_level=True`៖


In [2]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

tokenizer = keras.preprocessing.text.Tokenizer(char_level=True,lower=False)
tokenizer.fit_on_texts([x['title'].numpy().decode('utf-8') for x in ds_train])

យើងក៏ចង់ប្រើតួអក្សរពិសេសមួយដើម្បីបង្ហាញ **ចប់សន្ទស្សន៍** ដែលយើងនឹងហៅវា \<eos>។ មកដាក់វាដោយដៃទៅក្នុងវាក្យសព្ទ៖


In [3]:
eos_token = len(tokenizer.word_index)+1
tokenizer.word_index['<eos>'] = eos_token

vocab_size = eos_token + 1

ឥឡូវនេះ ដើម្បីកូដអត្ថបទទៅជាអក្សរលេខ ជួរដេក យើងអាចប្រើបាន៖


In [4]:
tokenizer.texts_to_sequences(['Hello, world!'])

[[48, 2, 10, 10, 5, 44, 1, 25, 5, 8, 10, 13, 78]]

## ការបណ្តុះបណ្តាល RNN បង្កើតស្លោកឲ្យបាន

វិធីដែលយើងនឹងបណ្តុះបណ្តាល RNN ដើម្បីបង្កើតស្លោកព័ត៌មាន គឺដូចខាងក្រោម។ នៅក្នុងជំហានមួយៗ យើងនឹងយកស្លោកមួយ ដែលនឹងត្រូវបញ្ចូលចូលទៅក្នុង RNN ហើយសម្រាប់តួអក្សរបញ្ចូលមួយៗ យើងនឹងស្នើឲ្យបណ្តាញបង្កើតតួអក្សរចេញបន្ទាប់៖

![Image showing an example RNN generation of the word 'HELLO'.](../../../../../translated_images/km/rnn-generate.56c54afb52f9781d.webp)

សម្រាប់តួអក្សរចុងក្រោយនៃខ្សែស្រឡាយយើង នឹងស្នើឲ្យបណ្តាញបង្កើត `<eos>` token។

ភាពខុសគ្នាចម្បងរវាង RNN បង្កើតដែលយើងកំពុងប្រើនៅទីនេះ គឺយើងនឹងយកលទ្ធផលចេញពីជំហាននីមួយៗរបស់ RNN មិនមែនពីកោសិការចុងក្រោយតែប៉ុណ្ណោះទេ។ នេះអាចធ្វើទៅបានដោយកំណត់ប៉ារ៉ាម៉ែត្រ `return_sequences` ទៅកាន់កោសិការនៃ RNN។

ដូច្នេះ ក្នុងការបណ្តុះបណ្តាលផ្នែកបញ្ចូលទៅបណ្តាញនឹងជាខ្សែស្រឡាយតួអក្សរដែលបានកូដមួយចំនួននៃប្រវែងមួយ ហើយលទ្ធផលនឹងជាខ្សែស្រឡាយដែលមានប្រវែងដូចគ្នា ប៉ុន្តែប្រែប្រួលដោយផ្លាស់ទីមួយធាតុ ហើយបញ្ចប់ដោយ `<eos>`។ Minibatch នឹងមានខ្សែស្រឡាយប៉ុន្មានដង ដូច្នេះយើងត្រូវប្រើ **padding** ដើម្បីតម្រឹមខ្សែស្រឡាយទាំងអស់។

ឲ្យយើងបង្កើតអនុគមន៍ដែលនឹងផ្លាស់ប្តូរព័ត៌មានសម្រាប់យើង។ ពីព្រោះយើងចង់បញ្ចូល padding នៅលើកម្រិត minibatch ជាមុនសិន យើងនឹងដំបូងធ្វើការ batch ដataset ដោយហៅ `.batch()` រួចបន្ទាប់មក `map` វាដើម្បីធ្វើការបម្លែង។ ដូច្នេះ អនុគមន៍បម្លែងនឹងទទួលបាន minibatch មួយពេញជា argument៖


In [5]:
def title_batch(x):
    x = [t.numpy().decode('utf-8') for t in x]
    z = tokenizer.texts_to_sequences(x)
    z = tf.keras.preprocessing.sequence.pad_sequences(z)
    return tf.one_hot(z,vocab_size), tf.one_hot(tf.concat([z[:,1:],tf.constant(eos_token,shape=(len(z),1))],axis=1),vocab_size)

មានអ្វីៗ​សំខាន់​ច្រើន​ដែល​យើងធ្វើនៅទីនេះ៖
* យើងចាប់ដកអត្ថបទពិតពី string tensor ដំបូង
* `text_to_sequences` ដំណើរការប្រែបញ្ជី string ទៅជា​បញ្ជី tensor ពីរៃងខុសៗគ្នា
* `pad_sequences` បន្ថែម padding ទៅលើ tensor ទាំងនោះ ដល់កម្ពស់តែមួយ
* ចុងក្រោយ យើងបំលែងតួអក្សរទាំងអស់ទៅជា one-hot និងធ្វើការ shifting និងបន្ថែម `<eos>` ផងដែរ។ យើងនឹងឃើញភ្លាមៗហេតុផលហេតុអ្វីយើងត្រូវការតួអក្សរត្រូវបានបំលែងជា one-hot encoded

ទោះជាយ៉ាងណា មុខងារនេះគឺជាមុខងារ **Pythonic** មានន័យថា វាមិនអាចបម្លែងដោយស្វ័យប្រវត្តិទៅជា Tensorflow computational graph បានទេ។ យើងនឹងទទួលបានកំហុស ប្រសិនបើយើងខំប្រើមុខងារនេះដោយផ្ទាល់ក្នុង `Dataset.map` function។ យើងត្រូវបង្រួមការហៅ Pythonic នេះដោយប្រើ `py_function` wrapper:


In [6]:
def title_batch_fn(x):
    x = x['title']
    a,b = tf.py_function(title_batch,inp=[x],Tout=(tf.float32,tf.float32))
    return a,b

> **កំណត់ចំណាំ**: ការបែងចែករវាងមុខងារបម្លែង Pythonic និង Tensorflow អាចហាក់ដូចជាស្មុគស្មាញចំពោះមួយភាគតិច ហើយអ្នកអាចកំពុងសួរថាហេតុអ្វីយើងមិនបម្លែងឈុតទិន្នន័យដោយប្រើមុខងារពី Python តាមទម្លាប់មុនពេលផ្ទុកទៅកាន់ `fit` ទេ។ ខណៈដែលវាអាចធ្វើបានប្រាកដ មុខងារ `Dataset.map` មានអត្ថប្រយោជន៍យ៉ាងខ្លាំង ពីព្រោះបំពង់បម្លែងទិន្នន័យត្រូវបានអនុវត្តដោយប្រើតង់ស័រហ្វ្លូក្រិហ្វកំណត់គណនា ដែលអាចប្រើអត្ថប្រយោជន៍នៃកំណត់គណនាការជាGPU ហើយកាត់បន្ថយការចាំបាច់ផ្ទុកទិន្នន័យរវាងCPU/GPU។

ឥឡូវនេះយើងអាចបង្កើតបណ្ដាញកំណើតរបស់យើង និងចាប់ផ្តើមបណ្តុះបណ្តាល។ វាអាចផ្អែកលើកោសិកា recurrent មួយណាមួយដែលយើងបានពិភាក្សានៅឯកត្តាមុន (សាមញ្ញ, LSTM ឬ GRU)។ ក្នុងឧទាហរណ៍របស់យើង យើងនឹងប្រើ LSTM។

ដោយសារតែបណ្ដាញទទួលតួអក្សរជា input ហើយទំហំវាកាបូប៉ុន្មានតូច យើងមិនត្រូវការជាន់ embedding ទេ ការបញ្ចូលតួតែតែមួយ-hot-encoded អាចចូលទៅកោសិកា LSTM ដោយផ្ទាល់។ ជាន់ output នឹងជា `Dense` ប្រភេទចម្រាញ់ ដែលនឹងបម្លែងចេញពី LSTM ទៅជាលេខកូដតួតែតែមួយ-hot។

បន្ថែមពីនេះ ពេលដែលយើងកំពុងដំណើរការជាមួយសេរីពហុបំណែង មួយ អាចប្រើជាន់ `Masking` ដើម្បីបង្កើតម៉ាស ដែលនឹងមិនគិតផ្នែកដែលពុម្ភបន្ថែមនៅចុងជួរខ្សែអក្សរ។ នេះមិនចាំបាច់តឹងត្រាប់ទេ ពីព្រោះយើងមិនសម្លឹងទៅលើអ្វីៗទាំងអស់ដែលនៅក្រៅតួ `<eos>` ទេ ប៉ុន្តែក្នុងករណីនេះយើងនឹងប្រើវាសម្រាប់ទទួលបានបទពិសោធន៍ជាមួយប្រភេទជាន់នេះ។ `input_shape` នឹងមាន `(None, vocab_size)` ដែល `None` មានន័យថាជាសំណុំជួរចំណងជើងបញ្ចេញផ្សេងៗ ហើយទំហំនៃប្រអប់គឺ `(None,vocab_size)` ផងដែរ ដូចដែលអ្នកអាចមើលឃើញពី `summary`:


In [7]:
model = keras.models.Sequential([
    keras.layers.Masking(input_shape=(None,vocab_size)),
    keras.layers.LSTM(128,return_sequences=True),
    keras.layers.Dense(vocab_size,activation='softmax')
])

model.summary()
model.compile(loss='categorical_crossentropy')

model.fit(ds_train.batch(8).map(title_batch_fn))

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
masking (Masking)            (None, None, 84)          0         
_________________________________________________________________
lstm (LSTM)                  (None, None, 128)         109056    
_________________________________________________________________
dense (Dense)                (None, None, 84)          10836     
Total params: 119,892
Trainable params: 119,892
Non-trainable params: 0
_________________________________________________________________
15000/15000 [==============================] - 229s 15ms/step - loss: 1.5385


## ការបង្កើតលទ្ធផល

ឥឡូវនេះដែលយើងបានបណ្តុះម៉ូដែលហើយ យើងចង់ប្រើវាដើម្បីបង្កើតលទ្ធផលមួយចំនួន។ ជាការដំបូង យើងត្រូវការរបៀបមួយក្នុងការបកប្រែអក្សរដែលតំណាងដោយលំដាប់លេខសញ្ញាសម្គាល់។ ដើម្បីធ្វើការនេះ យើងអាចប្រើមុខងារ `tokenizer.sequences_to_texts` ប៉ុន្តែវាមិនដំណើរការល្អជាមួយការបំបែកសញ្ញាលេខនៅកម្រិតតួអក្សរទេ។ ដូច្នេះ យើងនឹងយកវចនានុក្រមនៃសញ្ញាសម្គាល់ពី tokenizer (ហៅថា `word_index`), សង់ផែនទីត្រឡប់ក្រោយមួយ, ហើយសរសេរមុខងារបកប្រែរបស់យើងផ្ទាល់៖


In [10]:
reverse_map = {val:key for key, val in tokenizer.word_index.items()}

def decode(x):
    return ''.join([reverse_map[t] for t in x])

ឥឡូវនេះ យើងនឹងធ្វើការបង្កើត។ យើងនឹងចាប់ផ្តើមដោយខ្សែអក្សរ `start` មួយ ប្រើបំលែងវាទៅជាមួក `inp` ហើយបន្ទាប់មកនៅគ្រប់ជំហានយើងនឹងហៅបណ្ដាញរបស់យើងដើម្បីប៉ាន់ស្មានតួអក្សរបន្ទាប់។ 

ចេញពីបណ្ដាញ `out` គឺជាវ៉ែថ័រមានធាតុ `vocab_size` ដែលតំណាងឱ្យប្រសិទ្ធភាពនៃតួអក្សរនីមួយៗ ហើយយើងអាចស្វែងរកលេខតួអក្សរដែលមានប្រសិទ្ធភាពខ្ពស់បំផុតដោយប្រើ `argmax`។ បន្ទាប់មក យើងបន្ថែមតួអក្សរនោះទៅក្នុងបញ្ជីតួអក្សរដែលបានបង្កើត ហើយបន្តការបង្កើត។ ដំណើរការនៃការបង្កើតតួអក្សរមួយនេះត្រូវបានធ្វើឡើងជាប្រសិទ្ធិភាព `size` ដងដើម្បីបង្កើតចំនួនតួអក្សរត្រូវការ ហើយយើងបញ្ចប់មុនពេលរបស់វានៅពេលជួប `eos_token`។


In [12]:
def generate(model,size=100,start='Today '):
        inp = tokenizer.texts_to_sequences([start])[0]
        chars = inp
        for i in range(size):
            out = model(tf.expand_dims(tf.one_hot(inp,vocab_size),0))[0][-1]
            nc = tf.argmax(out)
            if nc==eos_token:
                break
            chars.append(nc.numpy())
            inp = inp+[nc]
        return decode(chars)
    
generate(model)

'Today #39;s lead to strike for the strike for the strike for the strike (AFP)'

## លទ្ធផលសម្រាប់ការយកមាតិកាត្រង់ពេលបណ្តុះបណ្តាល

ដោយសារតែយើងមិនមានមាត្រដ្ឋានណាមួយដែលមានប្រយោជន៍ដូចជា *ភាពត្រឹមត្រូវ* ទេ វិធីតែមួយដែលយើងអាចមើលឃើញថាគំរូរបស់យើងកាន់តែប្រសើរឡើងគឺដោយ **យកមាតិកា** ខ្សែអក្សរ​ដែលបង្កើតឡើងក្នុងពេលបណ្តុះបណ្តាល។ ដើម្បីបំពេញការនេះ យើងនឹងប្រើ **callbacks** ឧ. អនុគមន៍ដែលយើងអាចផ្តល់ទៅអនុគមន៍ `fit` ហើយវានឹងត្រូវបានហៅជាបន្តបន្ទាប់ក្នុងកំឡុងពេលបណ្តុះបណ្តាល។


In [13]:
sampling_callback = keras.callbacks.LambdaCallback(
  on_epoch_end = lambda batch, logs: print(generate(model))
)

model.fit(ds_train.batch(8).map(title_batch_fn),callbacks=[sampling_callback],epochs=3)

Epoch 1/3
15000/15000 [==============================] - 226s 15ms/step - loss: 1.2703
Today #39;s a lead in the company for the strike
Epoch 2/3
15000/15000 [==============================] - 227s 15ms/step - loss: 1.2057
Today #39;s the Market Service on Security Start (AP)
Epoch 3/3
15000/15000 [==============================] - 226s 15ms/step - loss: 1.1752
Today #39;s a line on the strike to start for the start


ឧទាហរណ៍នេះបានបង្កើតអត្ថបទល្អមួយចំនួនរួចហើយ ប៉ុន្តែវាអាចត្រូវបានបង្កើនកាន់តែប្រសើរឡើងនៅក្នុងវិធីជាច្រើន៖
* **អត្ថបទច្រើនជាងនេះ**។ យើងបានប្រើតែចំណងជើងសម្រាប់ភារកិច្ចរបស់យើងប៉ុណ្ណោះ ប៉ុន្តែអ្នកអាចចង់សាកល្បងជាមួយអត្ថបទពេញលេញ។ ចូរចាំថា RNNs មិនខ្លាំងណាស់ក្នុងការដោះស្រាយបន្ទាត់វែងៗពីរណោះ ដូច្នេះវាជាសមរម្យក្នុងការបំបែកពួកវាចូលទៅជាឃ្លាសង្ខេបឬជាប្រចាំហ្វឹកហាត់នៅលើរយៈពេលជាកំណត់ដែលបានកំណត់ជាមុន `num_chars` (ឧ. ២៥៦)។ អ្នកអាចសាកល្បងបម្លែងឧទាហរណ៍ខាងលើទៅជារចនាសម្ព័ន្ធបែបនេះដោយប្រើ [មេរៀនផ្លូវការ Keras](https://keras.io/examples/generative/lstm_character_level_text_generation/) ជាការបំផុសគំនិត។
* **LSTM ជាច្រើនស្រទាប់**។ វាផ្តល់ន័យក្នុងការសាកល្បង ២ ឬ ៣ ស្រទាប់នៃកោសិកា LSTM។ ដូចដែលយើងបានរៀបរាប់នៅក្នុងមុខវិជ្ជាពេលមុន ស្រទាប់នីមួយៗនៃ LSTM នឹងដកស្រង់គំរូខ្លះៗពីអត្ថបទ ហើយសម្រាប់អ្នកបង្កើតអត្ថបទតាមកម្រិតតួអក្សរ យើងអាចរំពឹងថា ស្រទាប់ LSTM កម្រិតទាបនឹងទទួលខុសត្រូវក្នុងការដកស្រង់ពាក្យបញ្ជប់ខ្លះៗ ហើយស្រទាប់ខ្ពស់ជាងនេះ - សម្រាប់ពាក្យ និងការរួមបញ្ចូលនៃពាក្យ។ វាអាចអនុវត្តបានយ៉ាងសាមញ្ញដោយផ្តល់ប៉ារ៉ាម៉ែត្រពីចំនួនស្រទាប់ទៅកាន់ការសង់ LSTM។
* អ្នកក៏អាចចង់សាកល្បងជាមួយ **ឯកតា GRU** ហើយមើលថាតើឯកតាណាអនុវត្តល្អជាង និងជាមួយ **ទំហំស្រទាប់លាក់ផ្សេងៗ**។ ទំហំស្រទាប់លាក់ធំពេកអាចនាំឲ្យមានការបំផ្លាញ (ឧ. បណ្ដាញនឹងរៀនអត្ថបទជាក់លាក់) ហើយទំហំតូចជាងអាចមិនបង្កើតលទ្ធផលល្អបាន។


## ការបង្កើតអត្ថបទទន់និងសីតុណ្ហភាព

នៅក្នុងការបញ្ជាក់ទ្រឹស្តី `generate` មុននេះ មនុស្សយើងបានយកតួអក្សរដែលមានប្រតិបត្តិភាពខ្ពស់បំផុតជាតួអក្សរបន្ទាប់ក្នុងអត្ថបទដែលបានបង្កើត។ នេះបណ្តាលឲ្យអត្ថបទមួយចំនួន "វិលជាថ្មី" រវាងលំដាប់តួអក្សរដូចគ្នាឡើងវិញ ជាញឹកញាប់ ដូចក្នុងឧទាហរណ៍នេះ៖
```
today of the second the company and a second the company ...
```

ដូច្នេះ ប្រសិនបើយើងមើលការបែងចែកប្រតិបត្តិភាពសម្រាប់តួអក្សរបន្ទាប់ វាអាចជា ភាពខុសគ្នារវាងប្រតិបត្តិភាពខ្ពស់បំផុតតិចណាស់ ឧទាហរណ៍ តួអក្សរមួយអាចមានប្រតិបត្តិភាព ០.២ ខណៈតួអក្សរមួយផ្សេងទៀត - ០.១៩ លី។ ឧទាហរណ៍ពេលស្វែងរកតួអក្សរបន្ទាប់ក្នុងលំដាប់ '*play*', តួអក្សរបន្ទាប់អាចជាស្ពាន (space) ឬ **e** (ដូចក្នុងពាក្យ *player*) បានស្មើ។

នេះនាំឲ្យយើងសន្និដ្ឋានថា មិនមែនជារឿង "ត្រឹមត្រូវ" ទៅតែងតួអក្សរដែលមានប្រតិបត្តិភាពខ្ពស់ជានិច្ចទេ ពីព្រោះការជ្រើសរើសលេខពីរខ្ពស់បំផុតអាចនាំឲ្យយើងទទួលបានអត្ថបទមានអត្ថន័យទៀត។ វាជាការយល់ឃើញល្អក្នុងការធ្វើការគំរូបតួអក្សរពីការបែងចែកប្រតិបត្តិភាពដែលបានផ្ដល់ដោយលទ្ធផលបណ្តាញ។

ការគំរូបនេះអាចធ្វើបានដោយប្រើមុខងារ `np.multinomial` ដែលអនុវត្តការបែងចែកប្រតិបត្តិភាពហៅថា **multinomial distribution**។ មុខងារដែលអនុវត្តការបង្កើតអត្ថបទប្រភេទ**ទន់**នេះត្រូវបានកំណត់ខាងក្រោម៖


In [33]:
def generate_soft(model,size=100,start='Today ',temperature=1.0):
        inp = tokenizer.texts_to_sequences([start])[0]
        chars = inp
        for i in range(size):
            out = model(tf.expand_dims(tf.one_hot(inp,vocab_size),0))[0][-1]
            probs = tf.exp(tf.math.log(out)/temperature).numpy().astype(np.float64)
            probs = probs/np.sum(probs)
            nc = np.argmax(np.random.multinomial(1,probs,1))
            if nc==eos_token:
                break
            chars.append(nc)
            inp = inp+[nc]
        return decode(chars)

words = ['Today ','On Sunday ','Moscow, ','President ','Little red riding hood ']
    
for i in [0.3,0.8,1.0,1.3,1.8]:
    print(f"\n--- Temperature = {i}")
    for j in range(5):
        print(generate_soft(model,size=300,start=words[j],temperature=i))


--- Temperature = 0.3
Today #39;s strike #39; to start at the store return
On Sunday PO to Be Data Profit Up (Reuters)
Moscow, SP wins straight to the Microsoft #39;s control of the space start
President olding of the blast start for the strike to pay &lt;b&gt;...&lt;/b&gt;
Little red riding hood ficed to the spam countered in European &lt;b&gt;...&lt;/b&gt;

--- Temperature = 0.8
Today countie strikes ryder missile faces food market blut
On Sunday collores lose-toppy of sale of Bullment in &lt;b&gt;...&lt;/b&gt;
Moscow, IBM Diffeiting in Afghan Software Hotels (Reuters)
President Ol Luster for Profit Peaced Raised (AP)
Little red riding hood dace on depart talks #39; bank up

--- Temperature = 1.0
Today wits House buiting debate fixes #39; supervice stake again
On Sunday arling digital poaching In for level
Moscow, DS Up 7, Top Proble Protest Caprey Mamarian Strike
President teps help of roubler stepted lessabul-Dhalitics (AFP)
Little red riding hood signs on cash in Carter-youb

---

KeyError: 0

យើងបានណែនាំអង្កត់ផ្ចិតមួយទៀតដែលហៅថា **សីតុខ** ដែលប្រើសម្រាប់បង្ហាញពីរបៀបយើងគួរតែខិតខំយ៉ាងម៉េចក្នុងការបន្តតាមប្រាក់ប្រហែលខ្ពស់បំផុត។ ប្រសិនបើសីតុណ្ហភាពគឺ 1.0 យើងធ្វើការប៉ាន់ប្រមាណតាមម៉ុលទីណូម្យលាដោយសមរម្យ ហើយនៅពេលដែលសីតុណ្ហភាពទៅរកអនាម័យ - ប្រាក់ប្រហែលទាំងអស់នឹងមានតុល្យភាព ហើយយើងជ្រើសរើសតួអក្សរបន្ទាប់ដោយចៃដន្យ។ ក្នុងឧទាហរណ៍ខាងក្រោម យើងអាចសង្កេតឃើញថាអត្ថន័យអក្សរត្រូវបានបាត់បង់ពេលយើងបង្កើនសីតុណ្ហភាពច្រើនពេក ហើយវាបង្ហាញភាពស្រដៀងនឹងអត្ថបទដែលបានបង្កើតឡើងយ៉ាងម៉ត់ចត់ដែលមានលក្ខណៈ "ចំណោត" នៅពេលវាហាក់ដូចជាយឺនទៅកាន់ 0។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**៖  
ឯកសារ​នេះ​ត្រូវបាន​បកប្រែ​ដោយប្រើ​សេវាកម្ម​បកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ខណៈពេលយើងខំប្រឹងរកភាពត្រឹមត្រូវ សូមយល់ដឹងថា ការបកប្រែជា​អូតូម៉ាទិច​អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើម​នៅក្នុងភាសាម្ចាស់ផ្លូវគួរត្រូវបានចាត់ទុកជាមូលដ្ឋានដ៏មានអំណាច។ សម្រាប់ព័ត៌មានសំខាន់ វិជ្ជាជីវៈ​ដូចជាការបកប្រែ​ដោយមនុស្សមានជំនាញ​ត្រូវបានណែនាំ។ យើង​មិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុស ពីការប្រើប្រាស់ការបកប្រែនេះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
